In [ ]:

# ============================================
# Cell 0: Mount Google Drive & Copy Dataset to Temp Memory
# ============================================

import os
import json
import shutil
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split
from collections import defaultdict
import keras
from tensorflow.keras.layers import Conv2D, BatchNormalization, ReLU, Add, Input, UpSampling2D, Concatenate, AveragePooling2D, Dropout
from tensorflow.keras import losses
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, Callback
from tensorflow.keras.optimizers import AdamW
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# ============================================
# Step 1: Copy Dataset to Temp Memory
# ============================================

SOURCE_DIR = '/content/drive/MyDrive/unified'
TEMP_DIR = '/content/lane_data'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/lane_segment'

print('='*60)
print('Dataset Setup')
print('='*60)

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError(f'Dataset folder not found at: {SOURCE_DIR}')

print(f'📁 Source (Drive): {SOURCE_DIR}')
print(f'📁 Destination (Temp): {TEMP_DIR}')

if not os.path.isdir(TEMP_DIR):
    print('📋 Copying dataset to temp memory (this may take 2-3 minutes)...')
    shutil.copytree(SOURCE_DIR, TEMP_DIR)
    print('✅ Dataset copied to temp memory!')
else:
    print('✅ Dataset already exists in temp memory!')

DATASET_PATH = TEMP_DIR

print(f'✅ Dataset path for training: {DATASET_PATH}')
print(f'   frames: {len(os.listdir(os.path.join(DATASET_PATH, "frames")))}')
print(f'   masks: {len(os.listdir(os.path.join(DATASET_PATH, "lane-masks")))}')
print('='*60)

BEST_MODEL_PATH = os.path.join(DRIVE_SAVE_DIR, 'bestmodel.keras')
LATEST_MODEL_PATH = os.path.join(DRIVE_SAVE_DIR, 'latest.keras')
LOG_CSV_PATH = os.path.join(DRIVE_SAVE_DIR, 'training_log.csv')
HISTORY_JSON_PATH = os.path.join(DRIVE_SAVE_DIR, 'history.json')
STATUS_JSON_PATH = os.path.join(DRIVE_SAVE_DIR, 'status.json')

print(f'💾 Weights saved on Drive: {DRIVE_SAVE_DIR}')


# ============================================
# Cell 1: Data Preparation and Loading
# ============================================

print("\n" + "="*60)
print("Lane Segmentation Project with DeepLabV3+")
print("="*60)

IMG_SIZE = (640, 640)
BATCH_SIZE = 8
NUM_CLASSES = 2
VALID_IMAGE_EXTS = ('.png', '.jpg', '.jpeg')
MIN_LANE_RATIO = 1e-6
MAX_LANE_RATIO = 0.40

print("\n[1] Loading data...")

def file_stem(path):
    return os.path.splitext(os.path.basename(path))[0]

def collect_files_by_stem(folder):
    mapping = {}
    for name in os.listdir(folder):
        if name.lower().endswith(VALID_IMAGE_EXTS):
            mapping[file_stem(name)] = os.path.join(folder, name)
    return mapping

def decode_any_image(path, channels):
    data = tf.io.read_file(path)
    image = tf.io.decode_image(data, channels=channels, expand_animations=False)
    image.set_shape([None, None, channels])
    return image

def is_valid_training_pair(img_path, mask_path):
    if not (os.path.isfile(img_path) and os.path.isfile(mask_path)):
        return False, "missing_file"
    try:
        img = decode_any_image(img_path, 3)
        mask = decode_any_image(mask_path, 1)
        img_shape = tf.shape(img)
        mask_shape = tf.shape(mask)
        if int(img_shape[0]) != int(mask_shape[0]) or int(img_shape[1]) != int(mask_shape[1]):
            return False, "size_mismatch"
        if int(img_shape[0]) < 64 or int(img_shape[1]) < 64:
            return False, "too_small"
        mask_bin = tf.cast(mask > 127, tf.float32)
        lane_ratio = float(tf.reduce_mean(mask_bin).numpy())
        if lane_ratio <= MIN_LANE_RATIO:
            return False, "empty_mask"
        if lane_ratio > MAX_LANE_RATIO:
            return False, "mask_too_dense"
        return True, "ok"
    except Exception as exc:
        return False, f"decode_failed:{exc}"

all_image_paths = []
all_mask_paths = []
skipped_pairs = {}

frames_path = os.path.join(DATASET_PATH, 'frames')
masks_path = os.path.join(DATASET_PATH, 'lane-masks')

manifest_path = os.path.join(DATASET_PATH, 'valid_pairs.json')
if os.path.isfile(manifest_path):
    with open(manifest_path, encoding="utf-8") as f:
        manifest_pairs = json.load(f)
    print(f"   Manifest found: {len(manifest_pairs)} pre-validated pairs")
    for item in manifest_pairs:
        img_path = os.path.join(DATASET_PATH, item["image"])
        mask_path = os.path.join(DATASET_PATH, item["mask"])
        if os.path.isfile(img_path) and os.path.isfile(mask_path):
            all_image_paths.append(img_path)
            all_mask_paths.append(mask_path)
        else:
            skipped_pairs["missing_file"] = skipped_pairs.get("missing_file", 0) + 1
else:
    images_by_stem = collect_files_by_stem(frames_path)
    masks_by_stem = collect_files_by_stem(masks_path)
    skipped_pairs["image_without_mask"] = len(set(images_by_stem) - set(masks_by_stem))
    skipped_pairs["mask_without_image"] = len(set(masks_by_stem) - set(images_by_stem))
    candidate_pairs = [
        (images_by_stem[stem], masks_by_stem[stem])
        for stem in sorted(set(images_by_stem) & set(masks_by_stem))
    ]
    print(f"   Scanning folder: {len(candidate_pairs)} matching pairs — validating...")
    for img_path, mask_path in candidate_pairs:
        ok, reason = is_valid_training_pair(img_path, mask_path)
        if ok:
            all_image_paths.append(img_path)
            all_mask_paths.append(mask_path)
        else:
            skipped_pairs[reason] = skipped_pairs.get(reason, 0) + 1

if not all_image_paths:
    raise RuntimeError("No valid image/mask pairs found for training.")

print(f"   ✅ Valid pairs: {len(all_image_paths)}")
if skipped_pairs:
    print(f"   ⚠️ Skipped pairs: {skipped_pairs}")

print("\n[1.1] Splitting data without sequence frame leakage...")

def sequence_key(path):
    stem = file_stem(path)
    if stem.startswith('tusimple__'):
        rest = stem[len('tusimple__'):]
        return 'tusimple__' + rest.rsplit('_', 1)[0]
    if stem.startswith('custom__'):
        rest = stem[len('custom__'):]
        if rest.startswith('img'):
            num = ''.join(ch for ch in rest[3:] if ch.isdigit())
            bucket = int(num) // 50 if num else 0
            return f'custom__img_{bucket}'
        num = ''.join(ch for ch in rest if ch.isdigit())
        bucket = int(num) // 50 if num else 0
        return f'custom__num_{bucket}'
    return stem

groups = defaultdict(list)
for img_path, mask_path in zip(all_image_paths, all_mask_paths):
    groups[sequence_key(img_path)].append((img_path, mask_path))

group_names = sorted(groups.keys())
train_groups, val_groups = train_test_split(
    group_names, test_size=0.15, random_state=42
)

train_pairs, val_pairs = [], []
for name in train_groups:
    train_pairs.extend(groups[name])
for name in val_groups:
    val_pairs.extend(groups[name])

train_images = [p[0] for p in train_pairs]
train_masks = [p[1] for p in train_pairs]
val_images = [p[0] for p in val_pairs]
val_masks = [p[1] for p in val_pairs]

print(f"   ✅ Training sequences: {len(train_groups)} | Validation: {len(val_groups)}")
print(f"   ✅ Training images: {len(train_images)}")
print(f"   ✅ Validation images: {len(val_images)}")

print("\n[2] Setting up data loading and preprocessing...")

def load_image_and_mask(image_path, mask_path):
    img = decode_any_image(image_path, 3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)

    mask = decode_any_image(mask_path, 1)
    mask = tf.where(mask > 127, tf.cast(1, tf.uint8), tf.cast(0, tf.uint8))
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')

    return img, mask

def augment_geometric(image, mask):
    mask_float = tf.cast(mask, tf.float32)
    stacked = tf.concat([image, mask_float], axis=-1)

    stacked = tf.image.random_flip_left_right(stacked)

    angle = tf.random.uniform([], -0.1, 0.1)
    stacked = tf.image.rot90(stacked, k=tf.cast(angle * 10, tf.int32) % 4)

    h = tf.shape(stacked)[0]
    w = tf.shape(stacked)[1]
    scale = tf.random.uniform([], 0.85, 1.0)
    crop_h = tf.cast(tf.cast(h, tf.float32) * scale, tf.int32)
    crop_w = tf.cast(tf.cast(w, tf.float32) * scale, tf.int32)
    crop_h = tf.maximum(crop_h, 1)
    crop_w = tf.maximum(crop_w, 1)
    stacked = tf.image.random_crop(stacked, [crop_h, crop_w, 4])
    stacked = tf.image.resize(stacked, [h, w], method="nearest")

    max_dy = tf.maximum(tf.cast(tf.cast(h, tf.float32) * 0.08, tf.int32), 1)
    max_dx = tf.maximum(tf.cast(tf.cast(w, tf.float32) * 0.08, tf.int32), 1)
    stacked = tf.image.pad_to_bounding_box(stacked, max_dy, max_dx, h + 2 * max_dy, w + 2 * max_dx)
    offset_h = tf.random.uniform([], 0, 2 * max_dy + 1, dtype=tf.int32)
    offset_w = tf.random.uniform([], 0, 2 * max_dx + 1, dtype=tf.int32)
    stacked = tf.image.crop_to_bounding_box(stacked, offset_h, offset_w, h, w)

    augmented_image = stacked[:, :, :3]
    augmented_mask = tf.round(stacked[:, :, 3:])
    return augmented_image, tf.cast(augmented_mask, tf.uint8)

def augment_photometric(image, mask):
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.7, upper=1.3)
    image = tf.image.random_hue(image, max_delta=0.05)

    noise = tf.random.normal(tf.shape(image), mean=0.0, stddev=0.02)
    image = image + noise

    if tf.random.uniform([]) < 0.1:
        gray = tf.image.rgb_to_grayscale(image)
        image = tf.image.grayscale_to_rgb(gray)

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, mask

def augment_cutmix(image, mask):
    h, w, _ = tf.unstack(tf.shape(image))
    patch_h = tf.random.uniform([], 20, 80, dtype=tf.int32)
    patch_w = tf.random.uniform([], 20, 80, dtype=tf.int32)
    y = tf.random.uniform([], 0, h - patch_h, dtype=tf.int32)
    x = tf.random.uniform([], 0, w - patch_w, dtype=tf.int32)

    patch_mask = tf.ones([patch_h, patch_w, 1])
    padded_mask = tf.image.pad_to_bounding_box(patch_mask, y, x, h, w)
    padded_mask = tf.cast(padded_mask, tf.float32)

    noise_patch = tf.random.uniform([patch_h, patch_w, 3], 0, 255)
    padded_noise = tf.image.pad_to_bounding_box(noise_patch, y, x, h, w)

    image = image * (1 - padded_mask) + padded_noise * padded_mask
    return image, mask

def final_preprocess_for_resnet(image, mask):
    image = preprocess_input(image)
    return image, mask

def get_augmentation_pipeline():
    def pipeline(image, mask):
        image, mask = augment_geometric(image, mask)
        image, mask = augment_photometric(image, mask)
        if tf.random.uniform([]) < 0.3:
            image, mask = augment_cutmix(image, mask)
        return image, mask
    return pipeline

print("   ✅ Functions ready")

print("\n[3] Creating Dataset Pipeline...")

augmentation_pipeline = get_augmentation_pipeline()

train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks))
train_dataset = train_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augmentation_pipeline, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(final_preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=2000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks))
val_dataset = val_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.map(final_preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

print("   ✅ Dataset ready!")


# ============================================
# Cell 2: Model Architecture with Dropout
# ============================================

print("\n" + "="*60)
print("Building and Training Model")
print("="*60)

print("\n[4] Setting up Loss Functions...")

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_pred_probs = tf.nn.softmax(y_pred, axis=-1)
    y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=2, axis=-1)
    y_true_one_hot = tf.squeeze(y_true_one_hot, axis=-2)

    class_weights_tensor = tf.constant([1.0, 8.0], dtype=tf.float32)
    class_weights_tensor = tf.reshape(class_weights_tensor, [1, 1, 1, 2])

    intersection = tf.reduce_sum(y_true_one_hot * y_pred_probs * class_weights_tensor, axis=[1, 2])
    sum_true = tf.reduce_sum(y_true_one_hot * class_weights_tensor, axis=[1, 2])
    sum_pred = tf.reduce_sum(y_pred_probs * class_weights_tensor, axis=[1, 2])

    dice_coefficient = (2. * intersection) / (sum_true + sum_pred + smooth)
    mean_dice_coefficient = tf.reduce_mean(dice_coefficient)
    return 1. - mean_dice_coefficient

def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    y_pred_probs = tf.nn.softmax(y_pred, axis=-1)
    y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=2, axis=-1)
    y_true_one_hot = tf.squeeze(y_true_one_hot, axis=-2)

    ce = -y_true_one_hot * tf.math.log(y_pred_probs + 1e-7)
    pt = tf.where(y_true_one_hot == 1, y_pred_probs, 1 - y_pred_probs)
    focal = alpha * tf.pow(1 - pt, gamma) * ce
    return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))

def combined_loss(y_true, y_pred):
    focal = focal_loss(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return focal + 0.5 * dice

def dice_metric(y_true, y_pred):
    return 1.0 - dice_loss(y_true, y_pred)

print("   ✅ Loss Functions configured")

print("\n[5] Building DeepLabV3+ Architecture...")

def convolution_block(block_input, num_filters=256, kernel_size=3, dilation_rate=1, padding="same", use_bias=True):
    x = Conv2D(num_filters, kernel_size=kernel_size, dilation_rate=dilation_rate,
               padding="same", use_bias=use_bias)(block_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

def AtrousSpatialPyramidPooling(aspp_input):
    dims = aspp_input.shape
    x = AveragePooling2D(pool_size=(dims[1], dims[2]))(aspp_input)
    x = convolution_block(x, kernel_size=1, dilation_rate=1)
    out_pool = UpSampling2D(size=(dims[1], dims[2]), interpolation="bilinear")(x)

    out_1 = convolution_block(aspp_input, kernel_size=1, dilation_rate=1)
    out_6 = convolution_block(aspp_input, kernel_size=3, dilation_rate=6)
    out_12 = convolution_block(aspp_input, kernel_size=3, dilation_rate=12)
    out_18 = convolution_block(aspp_input, kernel_size=3, dilation_rate=18)

    x = Concatenate(axis=-1)([out_pool, out_1, out_6, out_12, out_18])
    output = convolution_block(x, kernel_size=1)
    return output

def DeeplabV3_OS16(image_size, num_classes=2):
    model_input = keras.Input(shape=(image_size, image_size, 3))

    base_model = keras.applications.ResNet50(weights='imagenet', include_top=False, input_tensor=model_input)

    x = base_model.get_layer("conv4_block6_out").output
    low_level_features = base_model.get_layer("conv2_block3_out").output

    for i in range(1, 4):
        shortcut = x

        x = Conv2D(512, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_1_conv")(x)
        x = BatchNormalization(name=f"custom_conv5_block{i}_1_bn")(x)
        x = ReLU(name=f"custom_conv5_block{i}_1_relu")(x)

        x = Conv2D(512, (3, 3), strides=(1, 1), padding="same", dilation_rate=2, name=f"custom_conv5_block{i}_2_conv")(x)
        x = BatchNormalization(name=f"custom_conv5_block{i}_2_bn")(x)
        x = ReLU(name=f"custom_conv5_block{i}_2_relu")(x)

        x = Conv2D(2048, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_3_conv")(x)
        x = BatchNormalization(name=f"custom_conv5_block{i}_3_bn")(x)

        if i == 1:
            shortcut = Conv2D(2048, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_0_conv")(shortcut)
            shortcut = BatchNormalization(name=f"custom_conv5_block{i}_0_bn")(shortcut)

        x = Add(name=f"custom_conv5_block{i}_add")([x, shortcut])
        x = ReLU(name=f"custom_conv5_block{i}_out")(x)

    encoder_output = x

    x = AtrousSpatialPyramidPooling(encoder_output)
    x = Dropout(0.2)(x)

    input_a = UpSampling2D(size=(4, 4), interpolation="bilinear")(x)
    input_b = convolution_block(low_level_features, num_filters=48, kernel_size=1)

    x = Concatenate(axis=-1)([input_a, input_b])
    x = convolution_block(x)
    x = Dropout(0.1)(x)
    x = convolution_block(x)
    x = UpSampling2D(size=(4, 4), interpolation="bilinear")(x)

    model_output = Conv2D(num_classes, kernel_size=(1, 1), padding="same")(x)

    return Model(inputs=model_input, outputs=model_output)

print("   ✅ Architecture built")

print("\n[6] Building model...")

model = DeeplabV3_OS16(image_size=IMG_SIZE[0], num_classes=NUM_CLASSES)

def mean_iou_metric(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

    confusion = tf.math.confusion_matrix(
        tf.reshape(y_true, [-1]),
        tf.reshape(y_pred, [-1]),
        num_classes=NUM_CLASSES,
        dtype=tf.float32
    )

    true_positives = tf.linalg.diag_part(confusion)
    false_positives = tf.reduce_sum(confusion, axis=0) - true_positives
    false_negatives = tf.reduce_sum(confusion, axis=1) - true_positives

    iou = true_positives / (
        true_positives + false_positives + false_negatives + 1e-7
    )

    return tf.reduce_mean(iou)

model.summary()

print("   ✅ Model built")


# ============================================
# Cell 3: Training with Single Progress Bar
# ============================================

print("\n[7] Setting up Callbacks...")

class SingleProgressBarCallback(Callback):
    """Single progress bar for each epoch with all metrics"""
    def __init__(self, total_steps=None):
        super().__init__()
        self.total_steps = total_steps
        self.epoch_start = None
        self.pbar = None
        self.current_epoch = 0

    def on_epoch_begin(self, epoch, logs=None):
        self.current_epoch = epoch + 1
        self.epoch_start = time.time()

        self.pbar = tqdm(
            total=self.total_steps,
            desc=f"Epoch {self.current_epoch}/{self.params['epochs']}",
            unit='batch',
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]',
            position=0,
            leave=True
        )

    def on_batch_end(self, batch, logs=None):
        if self.pbar:
            self.pbar.update(1)
            if logs:
                loss = logs.get('loss', 0)
                acc = logs.get('accuracy', 0)
                self.pbar.set_postfix({
                    'loss': f'{loss:.4f}',
                    'acc': f'{acc:.3f}'
                })

    def on_epoch_end(self, epoch, logs=None):
        if self.pbar:
            self.pbar.close()

        elapsed = time.time() - self.epoch_start

        print(f"\n{'='*70}")
        print(f"✅ Epoch {self.current_epoch}/{self.params['epochs']} completed in {elapsed:.1f}s")
        print(f"{'='*70}")
        print(f"   📉 Loss:          {logs.get('loss', 0):.6f}  |  Val Loss:     {logs.get('val_loss', 0):.6f}")
        print(f"   📈 Accuracy:      {logs.get('accuracy', 0):.4f}  |  Val Accuracy: {logs.get('val_accuracy', 0):.4f}")
        print(f"   📈 Dice:          {logs.get('dice_metric', 0):.4f}  |  Val Dice:     {logs.get('val_dice_metric', 0):.4f}")
        print(f"   📈 Mean IoU:      {logs.get('mean_iou_metric', 0):.4f}  |  Val Mean IoU: {logs.get('val_mean_iou_metric', 0):.4f}")
        print(f"   📉 Learning Rate: {float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)):.2e}")

        if hasattr(self, 'best_iou'):
            current_iou = logs.get('val_mean_iou_metric', 0)
            if current_iou > self.best_iou:
                self.best_iou = current_iou
                print(f"   🏆 New Best IoU: {self.best_iou:.4f} (saved)")
        else:
            self.best_iou = logs.get('val_mean_iou_metric', 0)
            if self.best_iou > 0:
                print(f"   🏆 Best IoU so far: {self.best_iou:.4f}")

        print(f"{'='*70}\n")

class DriveStatusCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.logs_history = {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            self.logs_history.setdefault(key, []).append(float(value))
        status = {
            'epoch': int(epoch) + 1,
            'logs': {key: float(value) for key, value in logs.items()},
            'history': self.logs_history,
            'best_model': BEST_MODEL_PATH,
            'latest_model': LATEST_MODEL_PATH,
        }
        with open(STATUS_JSON_PATH, 'w', encoding='utf-8') as f:
            json.dump(status, f, ensure_ascii=False, indent=2)
        with open(HISTORY_JSON_PATH, 'w', encoding='utf-8') as f:
            json.dump(self.logs_history, f, ensure_ascii=False, indent=2)

train_steps = len(train_images) // BATCH_SIZE
if len(train_images) % BATCH_SIZE != 0:
    train_steps += 1

callbacks = [
    ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor='val_mean_iou_metric',
        mode='max',
        verbose=0,
        save_best_only=True
    ),
    ModelCheckpoint(
        LATEST_MODEL_PATH,
        save_best_only=False,
        verbose=0
    ),
    CSVLogger(LOG_CSV_PATH, append=True),
    DriveStatusCallback(),
    SingleProgressBarCallback(total_steps=train_steps),
    EarlyStopping(
        monitor='val_mean_iou_metric',
        mode='max',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1,
        cooldown=2
    )
]

print("   ✅ Callbacks configured")
print(f"   Best weights: {BEST_MODEL_PATH}")
print(f"   Steps per epoch: {train_steps}")


# ============================================
# Training Stage Management
# ============================================

def is_backbone_layer(layer):
    return layer.name.startswith(('conv1', 'conv2', 'conv3', 'conv4', 'pool1'))

def compile_stage(model, stage, learning_rate, weight_decay):
    model.compile(
        optimizer=AdamW(learning_rate=learning_rate, weight_decay=weight_decay),
        loss=combined_loss,
        metrics=["accuracy", mean_iou_metric, dice_metric]
    )


print("\n" + "="*50)
print("Stage 1: Full Backbone Freeze (20 Epochs)")
print("="*50)

for layer in model.layers:
    if is_backbone_layer(layer):
        layer.trainable = False

print("✅ All backbone layers frozen")

compile_stage(model, stage=1, learning_rate=2e-4, weight_decay=1e-4)

history_stage1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=callbacks,
    verbose=0
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Setup
📁 Source (Drive): /content/drive/MyDrive/unified
📁 Destination (Temp): /content/lane_data
✅ Dataset already exists in temp memory!
✅ Dataset path for training: /content/lane_data
   frames: 4760
   masks: 4759
💾 Weights saved on Drive: /content/drive/MyDrive/lane_segment

Lane Segmentation Project with DeepLabV3+

[1] Loading data...
   Manifest found: 4759 pre-validated pairs
   ✅ Valid pairs: 4759

[1.1] Splitting data without sequence frame leakage...
   ✅ Training sequences: 27 | Validation: 5
   ✅ Training images: 3114
   ✅ Validation images: 1645

[2] Setting up data loading and preprocessing...
   ✅ Functions ready

[3] Creating Dataset Pipeline...


Epoch 10/20:  16%|█▌        | 61/390 [27:10<2:26:36, 26.74s/batch]


   ✅ Dataset ready!

Building and Training Model

[4] Setting up Loss Functions...
   ✅ Loss Functions configured

[5] Building DeepLabV3+ Architecture...
   ✅ Architecture built

[6] Building model...


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 640, 640,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 646, 646,  │          0 │ input_layer_5[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 320, 320,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 320, 320,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 320, 320,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 322, 322,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 160, 160,  │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 160, 160,  │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 160, 160,  │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 160, 160,  │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 160, 160,  │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 160, 160,  │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 160, 160,  │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 160, 160,  │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 160, 160,  │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 160, 160,  │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 160, 160,  │      1,024 │ conv2_block1_3_c

...


Stage 2: Selective Fine-Tuning (Last 11 Layers) (35 Epochs)
   ✅ Backbone: 11 layers trainable, 141 layers frozen
   📌 Trainable backbone layers (last 11):
      - conv4_block6_out
      - conv2d_50
      - conv2d_51
      - conv2d_52
      - conv2d_53
      - conv2d_54
      - conv2d_55
      - conv2d_56
      - conv2d_57
      - conv2d_58
      - conv2d_59


Epoch 1/35: 100%|██████████| 390/390 [03:58<00:00,  1.64batch/s]



✅ Epoch 1/35 completed in 238.2s
   📉 Loss:          0.048008  |  Val Loss:     0.084592
   📈 Accuracy:      0.9897  |  Val Accuracy: 0.9788
   📈 Dice:          0.9157  |  Val Dice:     0.8652
   📈 Mean IoU:      0.8674  |  Val Mean IoU: 0.7946
   📉 Learning Rate: 5.00e-06



Epoch 2/35:   2%|▏         | 8/390 [00:32<06:54,  1.09s/batch]

KeyboardInterrupt: 